# SIV-WAM 手撕学习 · 01：Dataset Core

本 Notebook 手写数据集适配层中最关键的纯函数：

```text
flat latent [F*H*W, C]
    ↓ restore
latent [F,H,W,C]
    ↓ camera concat
T-shape video / head-only mask
    ↓ permute
model input [C,F,H,W]
```

本节先不加载 LeRobot、不加载 Wan VAE、不加载真实模型。我们先把空间布局的核心逻辑手写并验证。

## 本节必须掌握的结论

- video 和 mask 的**布局类型**都是 `[B,C,F,H,W]`，但当前项目中它们的 H/W 可以不同；
- action 不是为了和 video/mask 保持同 shape，而是单独的结构化动作流 `[B,C,F,N,1]`；
- 三路视频的 T 形拼接：两个 wrist 先沿宽度拼接，再与 head 沿高度拼接；
- 当前 SIV mask 只使用 `cam_high`，所以 mask 不做 wrist 拼接。

In [51]:
from pathlib import Path

import torch

PROJECT_ROOT = Path.cwd()
REFERENCE_DATASET_FILE = (
    PROJECT_ROOT / 'siv_wam' /
    'dataset' / 'lerobot_latent_dataset.py'
)

print('torch:', torch.__version__)
print('reference exists:', REFERENCE_DATASET_FILE.exists())

torch: 2.9.0+cu126
reference exists: True


## 1. 手写 latent 字典展平

生产代码中的 `_flatten_latent_dict()` 不改变 tensor 内容，只把：

```python
{'observation.images.cam_high': {'latent': ...}}
```

变成：

```python
{'observation.images.cam_high.latent': ...}
```

In [52]:
def flatten_latent_dict(latent_dict):
    out = {}
    for key, value in latent_dict.items():
        if not isinstance(value, dict):
            raise TypeError(f'{key} must map to a dict')
        for inner_key, inner_value in value.items():
            out[f'{key}.{inner_key}'] = inner_value
    return out

nested = {
    'observation.images.cam_high': {
        'latent': torch.tensor([[1.0, 2.0]]),
        'latent_num_frames': 1,
    }
}
flat = flatten_latent_dict(nested)
print(flat)
assert 'observation.images.cam_high.latent' in flat
assert flat['observation.images.cam_high.latent'].shape == (1, 2)

{'observation.images.cam_high.latent': tensor([[1., 2.]]), 'observation.images.cam_high.latent_num_frames': 1}


## 2. 手写 `[F*H*W,C] → [F,H,W,C]`

这里先不使用 `einops.rearrange`，故意用 `reshape`，让元素顺序更直观。

In [53]:
def restore_flat_latent(flat_latent, latent_num_frames, latent_height, latent_width):
    if flat_latent.ndim != 2:
        raise ValueError(f'expected [F*H*W,C], got {tuple(flat_latent.shape)}')
    expected = latent_num_frames * latent_height * latent_width
    if flat_latent.shape[0] != expected:
        raise ValueError(
            f'flat length mismatch: got {flat_latent.shape[0]}, expected {expected}'
        )
    return flat_latent.reshape(
        latent_num_frames, latent_height, latent_width, flat_latent.shape[1]
    )

F, H, W, C = 2, 3, 4, 1
flat = torch.arange(F * H * W * C, dtype=torch.float32).reshape(F * H * W, C)
restored = restore_flat_latent(flat, F, H, W)
print('flat:', flat.shape)
print('restored:', restored.shape)
print('restored[...,0]:\n', restored[..., 0])
assert restored.shape == (F, H, W, C)
assert torch.equal(restored.flatten(0, 2), flat)

flat: torch.Size([24, 1])
restored: torch.Size([2, 3, 4, 1])
restored[...,0]:
 tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])


## 3. 手写 latent 归一化

生产代码使用：

```text
normalized = (latent - mean) / std
```

这里的 mean/std 可以是逐通道向量。

In [54]:
def normalize_latent(latent_fhwc, mean, std):
    mean = torch.as_tensor(mean, dtype=latent_fhwc.dtype).reshape(1, 1, 1, -1)
    std = torch.as_tensor(std, dtype=latent_fhwc.dtype).reshape(1, 1, 1, -1)
    if torch.any(std == 0):
        raise ValueError('std must be non-zero')
    return (latent_fhwc - mean) / std

x = torch.tensor([[[[2.0, 4.0]]]])
y = normalize_latent(x, mean=[1.0, 2.0], std=[1.0, 2.0])
print(y)
assert torch.allclose(y, torch.tensor([[[[1.0, 1.0]]]]))

tensor([[[[1., 1.]]]])


## 4. 手写三路视频的 T-shape 拼接

当前项目的空间布局：

```text
left wrist  [F, H_w, W_w, C] ─┐
right wrist [F, H_w, W_w, C] ─┘ 沿 dim=2 拼接 → [F,H_w,2W_w,C]

head        [F, H_h, 2W_w, C]
wrist       [F, H_w, 2W_w, C]
             沿 dim=1 拼接

最终 video  [F,H_h+H_w,2W_w,C]
```

In [55]:
def cat_video_latents(latent_dict, video_keys, mean=0.0, std=1.0, env_type='robotwin_tshape'):
    latent_list = []
    for key in video_keys:
        latent = restore_flat_latent(
            latent_dict[f'{key}.latent'],
            latent_dict[f'{key}.latent_num_frames'],
            latent_dict[f'{key}.latent_height'],
            latent_dict[f'{key}.latent_width'],
        )
        latent_list.append(normalize_latent(latent, mean, std))

    if env_type == 'robotwin_tshape':
        if len(latent_list) != 3:
            raise ValueError('robotwin_tshape expects head + two wrist latents')
        wrist = torch.cat(latent_list[1:], dim=2) # W
        return torch.cat([wrist, latent_list[0]], dim=1) # H

    return torch.cat(latent_list, dim=2)

def to_model_layout(latent_fhwc):
    return latent_fhwc.permute(3, 0, 1, 2).contiguous() # [C,F,H,W]


In [56]:
def make_flat_camera(F, H, W, C, value):
    return torch.full((F * H * W, C), float(value))

video_keys = [
    'observation.images.cam_high',
    'observation.images.cam_left_wrist',
    'observation.images.cam_right_wrist',
]

"""
cam_high: [2,4,6,2] 全 1
cam_left_wrist: [2,2,3,2] 全 2
cam_right_wrist: [2,2,3,2] 全 3
"""
video_data = {
    f'{video_keys[0]}.latent': make_flat_camera(2, 4, 6, 2, 1),
    f'{video_keys[0]}.latent_num_frames': 2,
    f'{video_keys[0]}.latent_height': 4,
    f'{video_keys[0]}.latent_width': 6,
    f'{video_keys[1]}.latent': make_flat_camera(2, 2, 3, 2, 2),
    f'{video_keys[1]}.latent_num_frames': 2,
    f'{video_keys[1]}.latent_height': 2,
    f'{video_keys[1]}.latent_width': 3,
    f'{video_keys[2]}.latent': make_flat_camera(2, 2, 3, 2, 3),
    f'{video_keys[2]}.latent_num_frames': 2,
    f'{video_keys[2]}.latent_height': 2,
    f'{video_keys[2]}.latent_width': 3,
}

video_fhwc = cat_video_latents(video_data, video_keys)
video_cfHW = to_model_layout(video_fhwc)
print('video F,H,W,C:', tuple(video_fhwc.shape))
print('video C,F,H,W:', tuple(video_cfHW.shape))

assert video_fhwc.shape == (2, 6, 6, 2)
assert video_cfHW.shape == (2, 2, 6, 6)
# wrist 的前半宽度来自 left wrist
assert torch.all(video_fhwc[:, :2, :3] == 2)
# wrist 的后半宽度来自 right wrist
assert torch.all(video_fhwc[:, :2, 3:] == 3)
# 后四行来自 head
assert torch.all(video_fhwc[:, 2:] == 1)

video F,H,W,C: (2, 6, 6, 2)
video C,F,H,W: (2, 2, 6, 6)


## 5. 手写当前配置下的 head-only mask 拼接

当前 SIV 配置通常只有：

```text
observation.masks.cam_high
```

因此不能把 mask 当作三路视频继续拼 T-shape。

In [57]:
def cat_mask_latents(latent_dict, mask_keys, mean=0.0, std=1.0, env_type='robotwin_tshape'):
    mask_list = []
    for key in mask_keys:
        latent = restore_flat_latent(
            latent_dict[f'{key}.latent'],
            latent_dict[f'{key}.latent_num_frames'],
            latent_dict[f'{key}.latent_height'],
            latent_dict[f'{key}.latent_width'],
        )
        mask_list.append(normalize_latent(latent, mean, std))

    if env_type == 'robotwin_tshape' and len(mask_list) == 1:
        return mask_list[0]
    if env_type == 'robotwin_tshape':
        wrist = torch.cat(mask_list[1:], dim=2)
        return torch.cat([wrist, mask_list[0]], dim=1)
    return torch.cat(mask_list, dim=2)

mask_key = 'observation.masks.cam_high'
mask_data = {
    f'{mask_key}.latent': make_flat_camera(2, 4, 6, 2, 9),
    f'{mask_key}.latent_num_frames': 2,
    f'{mask_key}.latent_height': 4,
    f'{mask_key}.latent_width': 6,
}
mask_fhwc = cat_mask_latents(mask_data, [mask_key])
mask_cfHW = to_model_layout(mask_fhwc)
print('mask F,H,W,C:', tuple(mask_fhwc.shape))
print('mask C,F,H,W:', tuple(mask_cfHW.shape))
assert mask_fhwc.shape == (2, 4, 6, 2)
assert mask_cfHW.shape == (2, 2, 4, 6)

mask F,H,W,C: (2, 4, 6, 2)
mask C,F,H,W: (2, 2, 4, 6)


## 6. 对照生产代码

请打开生产实现的以下函数，对照你刚才的手写版本：

```text
_flatten_latent_dict
_normalize_latent
_cat_video_latents
_cat_mask_latents
__getitem__ 中的 permute(3,0,1,2)
```

参考文件：
`C:\Users\10752\Documents\ChatGPT\WAM\remote_snapshot_20260918\siv_wam\dataset\lerobot_latent_dataset.py`

本 Notebook 的验收标准：

- 你能解释每一次 `cat` 的 dimension；
- 你能解释为什么 mask 单相机时不能拼 wrist；
- 所有断言通过；
- 你能在纸上写出视频和 mask 的最终 shape。

# 7. 手写 get_relative_pose()
RoboTwin 的左右末端位姿原本是绝对位姿：`[x,y,z,qx,qy,qz,qw]`。本项目把它们改成相对于窗口首帧的位姿：
```text
relative_translation = translation_t - translation_0
relative_rotation = rotation_0^{-1} * rotation_t
```
夹爪通道不经过这个函数。

In [58]:
import numpy as np
from scipy.spatial.transform import Rotation as R


def get_relative_pose(pose):
    """Convert [T,7] absolute xyz+xyzw pose to first-frame-relative pose."""
    if torch.is_tensor(pose):
        pose = pose.detach().cpu().numpy()
    pose = np.asarray(pose, dtype=np.float64)
    if pose.ndim != 2 or pose.shape[1] != 7:
        raise ValueError(f'expected [T,7], got {pose.shape}')

    rot = R.from_quat(pose[:, 3:7])
    first_rot = R.from_quat(np.repeat(pose[:1, 3:7], len(pose), axis=0))
    relative_trans = pose[:, :3] - pose[:1, :3]
    relative_quat = (first_rot.inv() * rot).as_quat()
    relative_pose = np.concatenate([relative_trans, relative_quat], axis=1)
    return torch.from_numpy(relative_pose).float()

left_abs = torch.tensor([
    [1.0, 2.0, 3.0, 0.0, 0.0, 0.0, 1.0],
    [1.2, 2.0, 3.1, 0.0, 0.0, 0.0, 1.0],
    [1.5, 2.2, 3.1, 0.0, 0.0, 0.0, 1.0],
])
left_rel = get_relative_pose(left_abs)
print(left_rel)
assert left_rel.shape == (3, 7)
assert torch.allclose(left_rel[0, :3], torch.zeros(3), atol=1e-6)
assert torch.allclose(left_rel[0, 3:], torch.tensor([0.,0.,0.,1.]), atol=1e-6)
assert torch.allclose(left_rel[1, :3], torch.tensor([.2,0.,.1]), atol=1e-6)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.2000, 0.0000, 0.1000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.5000, 0.2000, 0.1000, 0.0000, 0.0000, 0.0000, 1.0000]])


# 8. 手写 16 维物理动作到 30 维模型动作的映射
物理动作顺序是：
`left pose 7 | left gripper 1 | right pose 7 | right gripper 1`
模型使用 30 个通道，但真实动作只占这些位置：
```text
left pose     -> model channels 0..6
left gripper  -> model channel 28
right pose    -> model channels 7..13
right gripper -> model channel 29
```

In [59]:
ACTION_DIM = 30
USED_ACTION_CHANNEL_IDS = (
    list(range(0, 7)) + [28] + list(range(7, 14)) + [29]  # noqa: PIE808
)

def build_inverse_action_indices(action_dim=ACTION_DIM, used_ids=USED_ACTION_CHANNEL_IDS):
    inverse = [len(used_ids)] * action_dim # [16] * 30
    for physical_index, model_channel in enumerate(used_ids):
        inverse[model_channel] = physical_index
    return inverse

INVERSE_ACTION_IDS = build_inverse_action_indices()
print('used ids:', USED_ACTION_CHANNEL_IDS)
print('inverse ids:', INVERSE_ACTION_IDS)
assert len(USED_ACTION_CHANNEL_IDS) == 16
assert len(INVERSE_ACTION_IDS) == 30
assert INVERSE_ACTION_IDS[0] == 0
assert INVERSE_ACTION_IDS[28] == 7
assert INVERSE_ACTION_IDS[29] == 15

used ids: [0, 1, 2, 3, 4, 5, 6, 28, 7, 8, 9, 10, 11, 12, 13, 29]
inverse ids: [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 7, 15]


In [60]:
def map_physical_action_to_model(action_16, inverse_ids=INVERSE_ACTION_IDS):
    """Map [T,16] physical actions to [T,30] model actions."""
    action_16 = torch.as_tensor(action_16, dtype=torch.float32)
    if action_16.ndim != 2 or action_16.shape[1] != 16:
        raise ValueError(f'expected [T,16], got {tuple(action_16.shape)}')
    # The extra column is the zero filler used by unused model channels.
    action_17 = torch.nn.functional.pad(action_16, (0, 1), value=0.0)
    index = torch.as_tensor(inverse_ids, dtype=torch.long)
    return action_17[:, index]

physical = torch.arange(16, dtype=torch.float32).repeat(3, 1)
model_action = map_physical_action_to_model(physical)
print('physical:', physical.shape)
print('model:', model_action.shape)
print('model channel 0:', model_action[0, 0])
print('model channel 28:', model_action[0, 28])
print('model channel 14:', model_action[0, 14])
assert model_action.shape == (3, 30)
assert model_action[0, 0] == physical[0, 0]
assert model_action[0, 28] == physical[0, 7]
assert model_action[0, 29] == physical[0, 15]
assert torch.all(model_action[:, 14:28] == 0)

physical: torch.Size([3, 16])
model: torch.Size([3, 30])
model channel 0: tensor(0.)
model channel 28: tensor(7.)
model channel 14: tensor(0.)


# 9. 手写动作窗口的时间对齐与最终 reshape
对于 `latent_frame_ids`，生产代码先计算：
```text
act_shift = latent_frame_ids[0] - local_start_frame
frame_stride = latent_frame_ids[1] - latent_frame_ids[0]
required_action_num = latent_frame_num * frame_stride * 4
```
然后将 `[T,30]` 变成模型使用的 `[30,F,N,1]`。

In [61]:
def align_and_reshape_action(
    action_16, local_start_frame, latent_frame_ids,
    inverse_ids=INVERSE_ACTION_IDS,
):
    action_16 = torch.as_tensor(action_16, dtype=torch.float32)
    latent_frame_ids = torch.as_tensor(latent_frame_ids, dtype=torch.long)
    act_shift = int(latent_frame_ids[0] - local_start_frame)
    frame_stride = int(latent_frame_ids[1] - latent_frame_ids[0])
    action_16 = action_16[act_shift:]

    latent_frame_num = (len(latent_frame_ids) - 1) // 4 + 1
    required_action_num = latent_frame_num * frame_stride * 4
    action_16 = torch.nn.functional.pad(
        action_16, (0, 0, frame_stride * 4, 0), value=0.0
    )[:required_action_num]
    if action_16.shape[0] != required_action_num:
        raise ValueError('not enough action frames after alignment')

    action_30 = map_physical_action_to_model(action_16, inverse_ids)
    action_30 = action_30.reshape(latent_frame_num, frame_stride * 4, 30)
    action_30 = action_30.permute(2, 0, 1).unsqueeze(-1).contiguous()
    return action_30, {
        'act_shift': act_shift,
        'frame_stride': frame_stride,
        'latent_frame_num': latent_frame_num,
        'required_action_num': required_action_num,
    }

raw_action = torch.randn(100, 16)
latent_frame_ids = list(range(12, 12 + 4 * 17, 4))
aligned_action, info = align_and_reshape_action(
    raw_action, local_start_frame=10, latent_frame_ids=latent_frame_ids
)
print(info)
print('aligned action:', aligned_action.shape)
assert info == {
    'act_shift': 2,
    'frame_stride': 4,
    'latent_frame_num': 5,
    'required_action_num': 80,
}
assert aligned_action.shape == (30, 5, 16, 1)

{'act_shift': 2, 'frame_stride': 4, 'latent_frame_num': 5, 'required_action_num': 80}
aligned action: torch.Size([30, 5, 16, 1])


# 本节验收
完成后你应该能够解释：
1. 首帧相对位姿为什么是零平移和单位旋转；
2. 为什么夹爪通道不经过 `get_relative_pose()`；
3. 为什么物理动作只有 16 维，但模型动作是 30 维；
4. `act_shift、frame_stride、required_action_num` 分别解决什么问题；
5. 为什么最终 `action` 要变成 `[30,F,N,1]`。

# 第 10 节：完整复现 _action_post_process()
前面已经分别手写了相对位姿、动作通道映射和 reshape。本节把它们串成生产代码对应的完整动作后处理。
```text
原始 [T,16]
  ↓ 起点对齐
  ↓ 左右位姿变为相对位姿
  ↓ 前补 frame_stride*4 个动作位置
  ↓ 裁剪到 required_action_num
  ↓ 加零填充列，映射到 30 个模型通道
  ↓ q01/q99 归一化
  ↓ reshape
最终 [30,F,N,1] + [30,F,N,1] action_mask
```

In [62]:
import torch.nn.functional as F


def action_post_process_core(
    action, local_start_frame, latent_frame_ids,
    q01, q99, inverse_ids=INVERSE_ACTION_IDS,
):
    action = torch.as_tensor(action, dtype=torch.float32)
    latent_frame_ids = torch.as_tensor(latent_frame_ids, dtype=torch.long)
    if action.ndim != 2 or action.shape[1] != 16:
        raise ValueError(f'expected raw action [T,16], got {tuple(action.shape)}')
    if latent_frame_ids.numel() < 2:
        raise ValueError('at least two latent frame ids are required')

    act_shift = int(latent_frame_ids[0] - local_start_frame)
    frame_stride = int(latent_frame_ids[1] - latent_frame_ids[0])
    if act_shift < 0 or frame_stride <= 0:
        raise ValueError('invalid action/frame alignment')
    action = action[act_shift:]

    # robotwin_tshape: relative-transform only the two 7D pose blocks.
    left_pose = get_relative_pose(action[:, :7])
    right_pose = get_relative_pose(action[:, 8:15])
    action = torch.cat([
        left_pose, action[:, 7:8],
        right_pose, action[:, 15:16],
    ], dim=1)

    leading = frame_stride * 4
    action = F.pad(action, (0, 0, leading, 0), value=0.0)
    latent_frame_num = (len(latent_frame_ids) - 1) // 4 + 1
    required_action_num = latent_frame_num * frame_stride * 4
    action = action[:required_action_num]
    if action.shape[0] != required_action_num:
        raise ValueError('not enough action frames after alignment')

    # In the production implementation, the leading temporal padding is
    # marked valid; only the unused model channels are masked out below.
    action_mask = torch.ones_like(action, dtype=torch.bool)
    action_padded = F.pad(action, (0, 1), value=0.0)
    mask_padded = F.pad(action_mask, (0, 1), value=False)

    index = torch.as_tensor(inverse_ids, dtype=torch.long)
    action_aligned = action_padded[:, index]
    action_mask_aligned = mask_padded[:, index]

    q01 = torch.as_tensor(q01, dtype=action_aligned.dtype).reshape(1, -1)
    q99 = torch.as_tensor(q99, dtype=action_aligned.dtype).reshape(1, -1)
    if q01.shape[1] != 30 or q99.shape[1] != 30:
        raise ValueError('q01/q99 must have 30 channels')
    action_aligned = (action_aligned - q01) / (q99 - q01 + 1e-6) * 2.0 - 1.0

    N = frame_stride * 4
    action_aligned = action_aligned.reshape(latent_frame_num, N, 30)
    action_mask_aligned = action_mask_aligned.reshape(latent_frame_num, N, 30)
    action_aligned = action_aligned.permute(2, 0, 1).unsqueeze(-1).contiguous()
    action_mask_aligned = action_mask_aligned.permute(2, 0, 1).unsqueeze(-1).contiguous()
    action_aligned = action_aligned * action_mask_aligned

    info = {
        'act_shift': act_shift,
        'frame_stride': frame_stride,
        'latent_frame_num': latent_frame_num,
        'required_action_num': required_action_num,
        'leading_padding': leading,
    }
    return action_aligned, action_mask_aligned, info

In [63]:
raw_action = torch.zeros(120, 16, dtype=torch.float32)
# 给平移和夹爪放入可观察的数值；四元数保持单位旋转。
t = torch.arange(120, dtype=torch.float32)

# 左右手的平移轨迹
raw_action[:, 0] = t * 0.01
raw_action[:, 8] = -t * 0.01

# 左右手的单位四元数 [qx, qy, qz, qw]
raw_action[:, 3:7] = torch.tensor([0.0, 0.0, 0.0, 1.0])
raw_action[:, 11:15] = torch.tensor([0.0, 0.0, 0.0, 1.0])

# 左右夹爪
raw_action[:, 7] = 0.5
raw_action[:, 15] = 1.0

# 调试检查四元数范数
left_quat_norm = torch.linalg.norm(raw_action[:, 3:7], dim=1)
right_quat_norm = torch.linalg.norm(raw_action[:, 11:15], dim=1)

assert torch.allclose(
    left_quat_norm,
    torch.ones(120),
    atol=1e-6,
)

assert torch.allclose(
    right_quat_norm,
    torch.ones(120),
    atol=1e-6,
)

print("left quat norm:", left_quat_norm.unique())
print("right quat norm:", right_quat_norm.unique())

latent_frame_ids = list(range(12, 12 + 4 * 17, 4))
q01 = torch.zeros(30)
q99 = torch.ones(30)
action_out, action_mask_out, info = action_post_process_core(
    raw_action,
    local_start_frame=10,
    latent_frame_ids=latent_frame_ids,
    q01=q01,
    q99=q99,
)

print(info)
print('action_out:', tuple(action_out.shape))
print('action_mask_out:', tuple(action_mask_out.shape))
print('active channels at first frame:', torch.where(action_mask_out[:, 0, 0, 0])[0].tolist())
print('inactive channels at first frame:', torch.where(~action_mask_out[:, 0, 0, 0])[0].tolist())

assert action_out.shape == (30, 5, 16, 1)
assert action_mask_out.shape == (30, 5, 16, 1)
assert info['act_shift'] == 2
assert info['frame_stride'] == 4
assert info['latent_frame_num'] == 5
assert info['required_action_num'] == 80
assert torch.all(~action_mask_out[14:28])
assert torch.all(action_mask_out[:14])
assert torch.all(action_mask_out[28:])

left quat norm: tensor([1.])
right quat norm: tensor([1.])
{'act_shift': 2, 'frame_stride': 4, 'latent_frame_num': 5, 'required_action_num': 80, 'leading_padding': 16}
action_out: (30, 5, 16, 1)
action_mask_out: (30, 5, 16, 1)
active channels at first frame: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 28, 29]
inactive channels at first frame: [14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


# 本节最关键的理解
这里暴露了生产代码中的一个重要约定：
- 时间前补的零动作在当前实现中被标记为有效；
- 模型没有物理意义的通道 14~27 被 action_mask 标记为无效；
- 归一化是在 30 维模型通道上进行；
- 最终 [30,F,N,1] 才能直接进入 action scheduler 和 action tokenization。
请将本节的 `action_post_process_core()` 与生产代码 `_action_post_process()` 逐行对照。下一节我们将把这个 Notebook 的输出接到 Flow Matching：对 video、mask、action 分别加噪并构造 `latent_dict/mask_dict/action_dict`。